# Cognopolis · Урок M1 — реактивный житель-сборщик

Ты управляешь жителем игры **Cognopolis** не мышкой, а **кодом-агентом**. В этом уроке строим **реактивного** агента: на каждом шаге он смотрит на состояние мира и решает одно действие — цикл `observe → decide → act → wait`.

**Что построим.** Житель ходит по карте, собирает дерево и камень, а когда рюкзак полон — возвращается **домой**: дом сам принимает добычу на склад (авто-разгрузка). А ещё житель умеет читать своё **поручение** — задачу, которую ты выписал ему в Ратуше. «Реактивный» значит: решение принимается *из текущего состояния*, без заранее зашитого плана.

**Тир агента:** реактивный (майлстоун игры M1). **Дальше по курсу:** планировщик (M3), LLM-агент (M4).

> ⚙️ **Working-first.** Ноутбук рассчитан на прогон `Run all` без правок — нужно лишь задать `BASE_URL` живого мира и свой `COGNOPOLIS_TOKEN` (из Ратуши) мира (ячейка ниже). Учебная активность — в секции **«Задачи»**.

**Ссылки** (подставь адрес своего мира вместо `<BASE_URL>`):
- API-доки (Swagger, кнопка **Authorize**): `<BASE_URL>/docs`
- 👀 Смотреть за своим агентом в браузере: `<BASE_URL>/?token=<твой токен>` (read-only)
- Контракт: житель видит мир **только** через API (`cognopolis_client`).

## 1. Сетап

Ставим официальный клиент игры из git-URL (PyPI пока нет) и задаём адрес мира.

In [ ]:
%pip install -q "cognopolis-client @ git+https://github.com/ITrubnikov/Train_of_Thought-Cognopolis.git#subdirectory=client"

In [ ]:
import os

# ⬇️ ЖИВОЙ МИР COGNOPOLIS (публичный инстанс). Можно переопределить переменной COGNOPOLIS_URL.
BASE_URL = os.environ.get("COGNOPOLIS_URL", "https://kindomklaster.com")

# ⬇️ ТВОЙ ТОКЕН ДОСТУПА (полный доступ к твоему жителю).
#   1. Открой BASE_URL в браузере и зарегистрируйся (логин + пароль).
#   2. Ратуша → вкладка «аккаунт» → кнопка «копировать» — это твой токен.
#   3. Вставь его ниже в COGNOPOLIS_TOKEN (или задай одноимённую переменную окружения / секрет Colab).
TOKEN = os.environ.get("COGNOPOLIS_TOKEN", "")  # ← вставь сюда свой токен в кавычки, если не используешь env
assert TOKEN, f"Вставь токен: зарегистрируйся на {BASE_URL}, скопируй токен из Ратуши и задай COGNOPOLIS_TOKEN."
print("Мир:", BASE_URL)
# Тир M1 — реактивный: LLM не нужен (он появится на M4).

## 2. Разогрев — подключаемся своим токеном

У тебя уже есть аккаунт (ты зарегистрировался на `BASE_URL`). Сюда подставляем его **токен** — им агент действует, и по нему же можно смотреть за жителем в браузере. `get_character()` и `get_map()` — это «глаза» агента (восприятие).

In [ ]:
from cognopolis_client import Client, GameError

c = Client(BASE_URL, token=TOKEN)
print("👀 Смотри за жителем в браузере:", f"{BASE_URL}/?token={TOKEN}")

ch = c.get_character()
print("позиция:", (ch["x"], ch["y"]), "| рюкзак:", ch["inventory"], "| cap:", ch["inventory_cap"])

world = c.get_map()
nodes = sorted({t["content"] for t in world["tiles"]})
print("карта", world["size"], "×", world["size"], "| что есть на клетках:", nodes)

Каждое действие возвращает результат **и кулдаун** — сколько ждать до следующего хода. Это естественный ритм петли агента.

In [ ]:
# Одно действие: шаг вправо. В ответе видно новое состояние и cooldown.
res = c.move(ch["x"] + 1, ch["y"], reason="разогрев — пробую сходить")
print("cooldown:", res["cooldown"], "c | новая позиция:", (res["character"]["x"], res["character"]["y"]))
c.wait_cooldown()  # observe → decide → act → ВОТ ЭТО ОЖИДАНИЕ

## 3. Разбор — паттерн реактивного агента

Реактивный агент крутит один и тот же цикл и **каждый раз решает заново из текущего состояния**:

```
observe  →  decide  →  act  →  wait  →  (снова observe)
```

- **observe** — `get_character()` (где я, что в рюкзаке) + `get_map()` (где ноды) + `get_assignment()` (есть ли у меня **поручение** от игрока);
- **decide** — правило: *если рюкзак полон → иду **домой** (дом сам примет добычу на склад); иначе → иду к нужному ресурсу и `gather`*;
- **act** — одно действие: `move` (шаг к цели) / `gather`;
- **wait** — `wait_cooldown()` досыпает кулдаун, и цикл повторяется.

Два штриха мира M1:

1. **Разгрузка автоматическая.** Отдельного действия «сдать на склад» нет: как только житель **ступает на свой дом** `(0,0)`, рюкзак сам пересыпается на склад — в ответе `move` появляется поле `banked`. Возврат домой *и есть* разгрузка.
2. **Поручение — это задача, не приказ.** Игрок пишет его в Ратуше (вкладка «поручения»), агент забирает его сам через `get_assignment()` и **сам решает, как** выполнять (или честно игнорирует — тогда в «Жителях» загорится диагноз).

Ключевая мысль M1: **никакого заранее зашитого маршрута**. Поведение «рождается» из условий на состояние — поэтому при полном рюкзаке агент сам сворачивает к дому. Подробный разбор и трейсы — в лекции урока.

## 4. Задачи — собери своего реактивного сборщика

Ниже — **рабочий каркас**: вспомогательные функции + петля + функция решения `decide()`. Базовая версия `decide()` уже работает (всегда идёт к ближайшему дереву и рубит), но она «туповата». **Твоя задача — сделать её реактивной:**

1. **Разгрузка.** Если рюкзак полон (`carried >= inventory_cap`) — иди **домой** `HOME = (0, 0)`: дом сам примет добычу на склад (в ответе `move` будет поле `banked`).
2. **Баланс ресурсов.** Иначе бери тот ресурс, которого у тебя **меньше** (дерево vs камень) — тогда агент реально *выбирает маршрут* между разными нодами.
3. **Поручение.** Перед выбором ресурса спроси у Ратуши своё поручение (`c.get_assignment()`). Если тебе поручили добычу (`type == "gather"`) — добывай **порученный** ресурс, а не свой выбор. Выпиши себе поручение в браузере (Ратуша → вкладка «поручения») и посмотри, как агент его подхватывает — а в «Жителях» строка «Поручение vs Сейчас» покажет, слушается ли он.

Подсказки в коде помечены `# TODO`. Ноутбук исполняется и до, и после правок — улучшай постепенно.

In [ ]:
RESOURCE_NODE = {"wood": "tree", "stone": "rock"}  # ресурс -> клетка, которая его даёт
HOME = (0, 0)  # дом жителя: шаг на эту клетку авто-разгружает рюкзак на склад

def nearest(ch, tiles, content):
    """Ближайшая клетка с заданным содержимым (по манхэттенскому расстоянию)."""
    here = (ch["x"], ch["y"])
    nodes = [t for t in tiles if t["content"] == content]
    return min(nodes, key=lambda t: abs(t["x"] - here[0]) + abs(t["y"] - here[1]))

def adjacent(ch, tx, ty):
    """Сосед или та же клетка (действовать можно вплотную)."""
    return abs(ch["x"] - tx) + abs(ch["y"] - ty) <= 1

def carried_total(ch):
    return sum(ch["inventory"].values())

def scarcer_resource(ch):
    """Ресурс, которого у жителя меньше (рюкзак + склад) — кандидат в цель."""
    owned = {r: ch["inventory"].get(r, 0) + ch["stored"].get(r, 0) for r in RESOURCE_NODE}
    return min(RESOURCE_NODE, key=lambda r: owned[r])

In [ ]:
def decide(ch, world):
    """Верни кортеж (target_x, target_y, action, reason).
    action — что делать: "gather" (добыть, когда дойдём вплотную) или "home" (идти домой —
    дом сам разгрузит рюкзак, отдельного действия для этого нет).

    БАЗОВАЯ версия (работает, но не реактивная): всегда идём к ближайшему дереву и рубим.
    Доработай по «Задачам» выше.
    """
    # TODO 1 — разгрузка: если carried_total(ch) >= ch["inventory_cap"] — иди домой:
    #   return HOME[0], HOME[1], "home", "рюкзак полон — иду домой разгружаться"

    # TODO 2 — баланс: вместо жёсткого "wood" возьми want = scarcer_resource(ch)

    # TODO 3 — поручение: если игрок поручил добычу — исполняй её ресурс:
    #   a = c.get_assignment()["assignment"]
    #   if a and a["type"] == "gather" and a["resource"]:
    #       want = a["resource"]  # (reason подскажет зрителям: f"по поручению: {want}")
    want = "wood"
    node = nearest(ch, world["tiles"], RESOURCE_NODE[want])
    return node["x"], node["y"], "gather", f"добываю {want} (базовая версия — улучшь меня!)"


def step_toward(ch, tx, ty):
    """Один шаг к цели: сперва по X, потом по Y (карта без стен)."""
    if ch["x"] != tx:
        return ch["x"] + (1 if tx > ch["x"] else -1), ch["y"]
    return ch["x"], ch["y"] + (1 if ty > ch["y"] else -1)


def run(rounds=40):
    for _ in range(rounds):
        ch = c.get_character()                       # observe
        tx, ty, action, reason = decide(ch, world)   # decide
        if action == "gather" and adjacent(ch, tx, ty):
            try:                                     # act: у ноды — добыча
                c.gather(reason=reason)
            except GameError as e:
                print("  gather blocked:", e.code)   # напр. inventory_full → пора домой
        elif (ch["x"], ch["y"]) != (tx, ty):         # act: шаг к цели (нода или дом)
            res = c.move(*step_toward(ch, tx, ty), reason=reason)
            banked = res["result"].get("banked")
            if banked:                               # ступили на дом — рюкзак сам ушёл на склад
                print("  дом: авто-разгрузка", banked)
        c.wait_cooldown()                            # wait

run()

## 5. Проверка

Базовый критерий — житель что-то реально добыл. Когда доработаешь `decide()`, целься в **большее**: качай оба навыка и возвращайся домой с полным рюкзаком, чтобы суммарно собрать заметно больше ёмкости рюкзака (значит цикл «рюкзак полон → домой → авто-разгрузка» реально замкнулся).

In [ ]:
ch = c.get_character()
total = sum(ch["inventory"].values()) + sum(ch["stored"].values())
skills = {k: v["level"] for k, v in ch["skills"].items()}
print("навыки:", skills, "| рюкзак:", ch["inventory"], "| склад:", ch["stored"], "| всего собрано:", total)

assert total >= 5, "Цель: собрать ≥ 5 ресурсов. Проверь decide() и число rounds."
print("✅ базовая цель достигнута — теперь сделай агента реактивным (домой-разгрузка + баланс + поручение)")

## Наблюдаемость — смотри за умом своего агента

Открой в браузере `BASE_URL/?token=<TOKEN>` (ссылка напечатана в разогреве) — в одной вкладке крутится агент, в другой видно его шаги, **мысль-пузырь** (`reason`, который ты передаёшь в действия) и Хронику событий. Это и есть «1 житель = 1 агент»: ты программируешь поведение, а наблюдаешь его как живого жителя.

Ручной тык по API — `BASE_URL/docs` (кнопка **Authorize**, вставь токен один раз без префикса `Bearer`).